<a href="https://colab.research.google.com/github/MichelleThuo/PersonalizedLearningChatBot_V2/blob/main/PersonalizedLearningChatBot_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Dependencies

In [3]:
!pip install fastapi uvicorn transformers torch accelerate safetensors pinecone openai pydantic sentence-transformers pytube moviepy ffmpeg h5py

# Step 2: Import Required Libraries

In [4]:
import os
import torch
import pinecone
import openai
import json
import h5py
import random
import moviepy.editor as mp
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pytube import YouTube
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"

# Step 3: Load Hugging Face Model (Zephyr 7B) for Chatbot

In [5]:
model_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def chat_response(user_query):
    input_ids = tokenizer.encode(user_query, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(input_ids, max_length=150, temperature=0.7, pad_token_id=tokenizer.eos_token_id)

    return tokenizer.decode(output[0], skip_special_tokens=True)

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

# Step 4: Initialize Pinecone for Vector Search

In [6]:
from pinecone import Pinecone, ServerlessSpec

# Set up Pinecone API key
PINECONE_API_KEY = "PINECONE_API_KEY"  # Replace with your actual key

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if the index exists and delete it if necessary
index_name = "quickstart"

# Delete the index if it exists
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"Index '{index_name}' deleted.")

# Create a new index with the correct dimension (768)
pc.create_index(
    name=index_name,
    dimension=768,  # Corrected to match the dimension of DistilBERT embeddings
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index(index_name)

# Load Sentence Transformer for embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Index 'quickstart' deleted.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 5: Process Documents & Store in Pinecone

In [7]:
def process_text_document(doc_text):
    embeddings = embedding_model.encode(doc_text).tolist()
    doc_id = f"doc_{random.randint(1000,9999)}"

    index.upsert(vectors=[(doc_id, embeddings, {"text": doc_text})])

    return f"Document '{doc_id}' indexed successfully!"

# Step 6: Process Videos (Transcribe with Whisper + Store in Pinecone)

In [8]:
whisper_model = pipeline("automatic-speech-recognition", model="openai/whisper-base")

def process_video(video_url):
    yt = YouTube(video_url)
    video_path = yt.streams.filter(only_audio=True).first().download(filename="video.mp4")

    audio = mp.AudioFileClip(video_path)
    audio.write_audiofile("audio.wav")

    transcription = whisper_model("audio.wav")["text"]

    embeddings = embedding_model.encode(transcription).tolist()
    video_id = f"video_{random.randint(1000,9999)}"

    index.upsert(vectors=[(video_id, embeddings, {"text": transcription})])

    return f"Video '{video_id}' transcribed & indexed!"

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Device set to use cpu


# Step 7: Smart Search Across Videos & Documents

In [9]:
def search_knowledge_base(query):
    query_embedding = embedding_model.encode(query).tolist()
    results = index.query(queries=[query_embedding], top_k=3, include_metadata=True)

    if results["matches"]:
        return [match["metadata"]["text"] for match in results["matches"]]
    else:
        return ["No relevant results found."]

# Step 8: Generate Smart Quizzes Using AI

In [10]:
def generate_quiz_from_text(text_content):
    prompt = f"Create a multiple-choice question based on the following content: {text_content}"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(input_ids, max_length=250, temperature=0.7)

    return tokenizer.decode(output[0], skip_special_tokens=True)

# Step 9: Convert AI-Generated Quizzes to H5P for Moodle

In [11]:
def save_quiz_as_h5p(quiz_text, filename="quiz.h5p"):
    with h5py.File(filename, "w") as h5f:
        h5f.create_dataset("quiz_text", data=quiz_text.encode("utf-8"))
    return f"Quiz saved as {filename} for Moodle!"

In [12]:
# FastAPI app code written directly in the cell
with open('main.py', 'w') as f:
    f.write("""

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class UserQuery(BaseModel):
    query: str

def chat_response(query):
    return f"You asked: {query}"

def process_text_document(doc_text):
    return f"Processed document: {doc_text[:50]}..."

def process_video(video_url):
    return f"Processed video from URL: {video_url}"

def search_knowledge_base(query):
    return [f"Search result for {query}: Example result 1", "Example result 2"]

def generate_quiz_from_text(text):
    return f"Generated quiz for: {text[:50]}..."

def save_quiz_as_h5p(quiz_text):
    print(f"Quiz saved as H5P: {quiz_text}")

@app.post("/chat/")
async def chat(input: UserQuery):
    response = chat_response(input.query)
    return {"response": response}

@app.post("/upload_text/")
async def upload_text(doc_text: str):
    response = process_text_document(doc_text)
    return {"message": response}

@app.post("/upload_video/")
async def upload_video(video_url: str):
    response = process_video(video_url)
    return {"message": response}

@app.post("/search/")
async def search(input: UserQuery):
    results = search_knowledge_base(input.query)
    return {"results": results}

@app.post("/generate_quiz/")
async def generate_quiz(input: UserQuery):
    quiz_text = generate_quiz_from_text(input.query)
    save_quiz_as_h5p(quiz_text)
    return {"quiz": quiz_text, "message": "Quiz saved as H5P for Moodle"}
    """)

# Run the app using uvicorn
!python3 -m uvicorn main:app --host 0.0.0.0 --port 8000

INFO:     Started server process [12242]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [12242]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.11/asyncio/runners.py", line 190, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/asyncio/base_events.py", line 641, in run_until_complete
    self.run_forever()
  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
    self._run_once()
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once
    handle._run()
  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self.

In [14]:
import os
os.listdir()

['.config', 'main.py', '__pycache__', 'sample_data']

# Step 10: Create API with FastAPI

In [19]:
ngrok.kill()

In [18]:
!pip install fastapi uvicorn pyngrok nest-asyncio

import uvicorn
from pyngrok import ngrok
import nest_asyncio
import threading
import time

# Allow nested event loops in Colab
nest_asyncio.apply()

# Function to run FastAPI
def run():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

# Start FastAPI in a separate thread
thread = threading.Thread(target=run)
thread.start()

# Wait a few seconds for Uvicorn to start
time.sleep(3)

# Authenticate ngrok using your authtoken
!ngrok authtoken "authtoken"# Use actual authtoken

# Expose the FastAPI server via ngrok (explicitly define HTTP)
public_url = ngrok.connect(8000, "http")
print("Public URL:", public_url)

INFO:     Started server process [11125]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Public URL: NgrokTunnel: "https://4723-35-186-172-4.ngrok-free.app" -> "http://localhost:8000"
